<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP10.2A/MP10.2A_channel_and_electrode.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP10.2A · The Channel Across Its Height, and the Electrode Under It

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/")
PROJECT = (['mp102a.py', 'MP10.2A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP10.2A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex10.2-solid-oxide-cell", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex10.2-solid-oxide-cell", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp102a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp102a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP10.2A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex10.2's problem.py, by its name in Ex10.2

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

In the solid oxide electrolysis cell of Ex10.2 the steam must travel twice before it can be split: **across the fuel channel**, from the gas flowing along it to the surface of the fuel electrode, and **through the porous electrode**, 0.5 mm of nickel and ceramic, to the reaction sites at the electrolyte. Ex10.2 treated the channel as one-dimensional - the gas mixed across its height - and priced the electrode by a **gas-transport loss** with an estimated limiting current density. Where the steam that reaches the reaction sites runs low, the Nernst potential rises, the current falls, and if it runs out the nickel of the electrode can oxidise: the steam fraction **at the reaction site** is what a cell designer watches.

So three quantities are wanted at every cell voltage: **the cell current** (the polarisation curve of the cell), **the steam fraction leaving the channel** (what must be condensed and fed again), and **the steam fraction at the reaction site along the channel** (where the electrode starves). A stack model or a controller asks them again for every new voltage, so the model is run many times. That is where a fast model pays.

This mini project computes all three for any cell voltage between 1.15 and 1.35 V, with one network.

## A2 · The cell, and what is new against Ex10.2

The cell is Ex10.2's: 100 × 100 mm of electrode, a fuel channel 100 mm long and 1 mm high, 90 % steam and 10 % hydrogen fed at 1 m/s, the kinetics and the resistance of Ex10.2. The cell below draws it and prints its data.

**Three changes.** First, the steam fraction is resolved **across the channel's height**, $y(x, z)$, carried along by Poiseuille's profile and diffused across, as in Ex09.2. Second, the steam is followed **through the porous electrode** to the reaction at the electrolyte, by diffusion with an effective diffusivity $D_\mathrm{eff}$ (an estimate: the gas diffusivity reduced by the porosity and the tortuosity of the pores). Steam goes in and hydrogen comes out, one for one, and nothing reacts inside the electrode's pores, so the profile through the electrode is a straight line and the steam at the reaction site is

$$y_r = y_w - K_E\, i, \qquad K_E = \frac{\delta}{2F\,c\,D_\mathrm{eff}}$$

with $y_w$ the steam at the electrode's surface, $\delta$ = 0.5 mm the electrode's thickness and $i$ the local current density. Third, the **charge transfer is evaluated at the reaction site**: the Nernst potential at $y_r$, the activation and the ohmic losses of Ex10.2, and **no gas-transport loss term**, because the gas transport is now resolved. Ex10.2's limiting current is no longer an input: it is where $y_r$ reaches zero, $i_L = y_w/K_E$ = 3.94 A/cm² at the inlet, which is where the estimate of $D_\mathrm{eff}$ came from (Ex10.2 assumed 4.0).

The cell is held at **800 °C** (isothermal): the heat of Ex10.2 is left out, so that this project is about the gas. Every kinetic and transport value is Ex10.2's estimate or a new one, marked as such.

In [ ]:
# The cell of Ex10.2, and its data
fig = pb.draw_soec()                                     # Ex10.2's drawing: the channel, the porous fuel electrode, the electrolyte
plt.show()

rows = [("cell", f"{pb.L_CHANNEL*1e3:.0f} × {pb.W_CELL*1e3:.0f} mm of electrode, held at {mp.T_CELL-273.15:.0f} °C (isothermal)"),
        ("fuel channel", f"{pb.H_CHANNEL*1e3:.0f} mm high, {pb.U_GAS:.0f} m/s mean speed, Poiseuille's profile, c = {pb.C_TOT:.2f} mol/m³"),
        ("gas fed", f"{100*pb.Y_IN:.0f} % steam, {100*(1-pb.Y_IN):.0f} % hydrogen; D = {pb.D_GAS:.1e} m²/s (Ex10.2's estimate)"),
        ("fuel electrode", f"{mp.DELTA*1e3:.1f} mm thick, D_eff = {mp.D_EFF:.1e} m²/s (estimate)"),
        ("kinetics", f"i0 = {pb.I_0:.2f} A/cm², ASR = {mp.ASR_C:.2f} ohm cm² at 800 °C (Ex10.2's estimates)"),
        ("cell voltage", f"V = {mp.V_LOW:.2f} to {mp.V_HIGH:.2f} V, a network input")]
for k, v in rows:
    print(f"{k:>15}  {v}")

**What you should see.** Ex10.2's cross-section of the cell: the hydrogen (fuel) channel on top, the porous fuel electrode under it, the electrolyte, the air side. And the table: everything is Ex10.2's except the temperature, held at 800 °C, and the electrode's thickness and effective diffusivity, which are new.

In [ ]:
# The current density against the steam at the electrode's surface, at three cell voltages
y = np.linspace(0.02, 0.9, 300)                          # steam fraction at the electrode's surface
fig, ax = plt.subplots(figsize=(7, 3.6))
for V, c in zip(mp.V_TEST, ("tab:blue", "tab:green", "tab:red")):
    ax.plot(100 * y, mp.current_at(V, y)[0], color=c, label=f"{V:.2f} V, electrode resolved (this project)")
    ax.plot(100 * y, pb.current(V, y, mp.T_CELL), color=c, ls="--", lw=0.9, label=f"{V:.2f} V, Ex10.2's gas-transport loss")
ax.set_xlabel("steam fraction at the electrode's surface [%]"); ax.set_ylabel("current density [A/cm²]"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
for yy in (0.9, 0.5, 0.2):
    print(f"steam at the surface {100*yy:3.0f} %: current density at 1.27 V {float(mp.current_at(1.27, yy)[0]):.2f} A/cm² "
          f"(Ex10.2's 1-D charge transfer {float(pb.current(1.27, yy, mp.T_CELL)):.2f}); "
          f"steam at the reaction site {yy - mp.K_E * float(mp.current_at(1.27, yy)[0]):.3f}")

**What you should see.** At every voltage the current density falls as the steam at the electrode's surface falls, steeply below about 20 %: the reaction site runs short of steam first. With the electrode resolved (solid) the current is lower than with Ex10.2's gas-transport loss (dashed), most where there is most steam: at 1.27 V and 90 % steam on the surface, 1.46 against 1.78 A/cm²; at 50 %, 1.15 against 1.22; at 20 %, 0.71 against 0.73. The difference is the hydrogen side of the electrode, which Ex10.2's loss term leaves out: the hydrogen made at the reaction site must diffuse out through the pores, so the site holds more hydrogen than the channel (43 % instead of 10 % at 90 % steam on the surface), which raises the Nernst potential. At 90 % steam on the surface the reaction site sees 0.57; at 20 %, only 0.04.

## A3 · The domain, the equations and their conditions

The domain is the channel in cross-section, $0 < x < L$ along it and $0 < z < h$ across the gap ($z = 0$ the electrode's surface, $z = h$ the interconnect), the velocity Poiseuille's $u(z) = 6U\,(z/h)(1 - z/h)$ at the mean speed $U$ = 1 m/s. The unknowns are the steam fraction $y(x, z)$ and the current density $i(x)$, both depending on the cell voltage $V$.

$$u(z)\,\frac{\partial y}{\partial x} = D\,\frac{\partial^2 y}{\partial z^2} \quad \text{in the channel}$$

| where | condition | why it holds |
|---|---|---|
| inlet ($x = 0$) | $y = 0.90$ | the feed: 90 % steam |
| electrode's surface ($z = 0$) | $D\,c\,\dfrac{\partial y}{\partial z} = \dfrac{i}{2F}$ | the steam diffuses into the electrode at the rate the current splits it (Faraday) |
| interconnect ($z = h$) | $\partial y/\partial z = 0$ | an impermeable steel wall |
| at every $x$ | $V = E(y_r) + \dfrac{RT}{F}\,\mathrm{asinh}\dfrac{i}{2 i_0} + i\,\mathrm{ASR}$, $\ y_r = y(x, 0) - K_E\, i$ | the charge transfer at the reaction site: the applied voltage covers the Nernst potential there, the activation loss and the ohmic loss |
| outlet ($x = L$) | none | diffusion along the channel is dropped, so the equation marches from the inlet |

The reductions are Ex09.2's and Ex10.2's, and the cell below prints the numbers. The exchange is one for one, so the molar concentration $c$ and the velocity do not change along the channel. The Reynolds number is of order one, so Poiseuille's profile holds from the inlet. Along the channel the Péclet number is 125, so diffusion along it is dropped (Ex10.2 kept it; it changes the outlet mean by less than a thousandth) and the equation is **parabolic in $x$**. The electrode's pores are thin against the channel's length, so the steam crosses them straight: their profile is linear across, with no march of its own.

In [ ]:
# The numbers behind the model
print(f"alpha = L D / (U h²) = {mp.ALPHA:.0f}: diffusion across the channel against convection along it")
print(f"beta  = h / (2 F c D) = {mp.BETA:.2e} per A/cm²: the slope of y across the channel that carries one A/cm²")
print(f"K_I = alpha beta = {mp.K_I:.3f} per A/cm²: the fall of the mean along the channel (Ex10.2's K_I = {pb.K_I:.3f})")
print(f"K_E = delta / (2 F c D_eff) = {mp.K_E:.3f} per A/cm²: the steam lost through the electrode")
print(f"limiting current at the inlet, y = 0.90: {pb.Y_IN / mp.K_E:.2f} A/cm² (Ex10.2 assumed {pb.I_LIM_0:.2f})")
print(f"Péclet number along the channel {pb.PE:.0f}; across it, U h / D = {pb.U_GAS*pb.H_CHANNEL/pb.D_GAS:.2f}")

**What you should see.** $\alpha$ = 80: the gas crosses the 1 mm gap by diffusion eighty times in the time it takes to flow along the channel, so it is nearly mixed across, but not quite. $\beta$ = 5.70e-03: at 1.5 A/cm² the steam fraction changes by about 0.003 across the channel. $K_E$ = 0.228: at the same current the reaction site sees 0.34 less steam than the electrode's surface, a hundred times more. The fall of the mean, $K_I$ = 0.456 per A/cm², is Ex10.2's.

---

## A4 · Why there is no exact solution

At every $x$ the current density is the root of the charge transfer at the reaction site, a nonlinear equation in the steam at the wall. The wall condition therefore depends on the unknown, and the current falls along the channel as the steam does: the profile across the channel changes along it, and separation of variables fails. No closed form exists, even for the one-dimensional model of Ex10.2.

What remains exact is the **balance of the mean**: integrated over the height with the flow's weight, the equation and its wall conditions say that the flow-weighted (mixing-cup) mean $\bar y$ falls at the rate the current splits steam,

$$\frac{d\bar y}{dX} = -K_I\, i, \qquad X = x/L$$

Ex10.2's steam balance without axial diffusion. If the gas were mixed across the channel ($y_w = \bar y$, **plug flow**) this would close the problem, with the current solved at every $X$ for $y_r = \bar y - K_E\,i$: the **hand calculation** of section B2. It keeps the electrode and misses only the profile across the channel; part C measures by how much.

**The zero-strength check.** The strength $S$ multiplies both transport resistances: the electrode's $K_E$ and the channel's resistance across, $1/\alpha$. This project is $S = 1$. As $S$ goes to zero the steam reaches the reaction site without loss, and the model becomes Ex10.2's one-dimensional steam balance and charge transfer at 800 °C, with no gas-transport loss and no axial diffusion: an ordinary differential equation, solved to rounding. Part C shows the solver approaching it in proportion to $S$.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the cell current, $I = \int i\,dA$ | the test voltages 1.18, 1.27 and 1.33 V, which you do not train on | within **0.1 %** of the ground truth |
| the outlet's mixing-cup steam fraction | the same | within **5e-04** of the ground truth |
| the steam fraction at the reaction site, beyond the first 5 mm | the same, along the channel | within **1e-03** of the ground truth everywhere |
| the time to answer a new voltage online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline, over the whole range of voltages. Its online answer - the current, the outlet mean and the steam at the reaction site along the channel for a voltage it has not seen - is timed against the reference model, which must march the channel again for every new voltage. The training time is reported apart, as the offline cost.

**Why these numbers.** The plug-flow hand calculation, which keeps the electrode but mixes the gas across the channel, misses by 0.13 to 0.19 % in the current, 5.0e-04 to 1.3e-03 in the outlet mean and 1.7e-03 to 3.0e-03 at the reaction site over the range of voltages (part C). The requirements are about half of that, so that the profile across the channel must be resolved; an engineering grid of 21 × 5 nodes meets them (part C). Ex10.2's one-dimensional model with its gas-transport loss overstates the current by 5 to 10 %: resolving the electrode is the larger correction, resolving the channel the finer one. The first 5 mm are left out of the reaction-site score, as in Ex09.2: at the inlet corner the flat feed meets the electrode's flux, and the steam at the surface drops by about 0.002 within the first half millimetre.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network is a function of three inputs and returns two numbers:

$$(X,\ Z,\ V)\ \longmapsto\ \big(y(X, Z; V),\ i(X; V)\big)$$

with $X = x/L$ and $Z = z/h$ the scaled position ($Z = 0$ the electrode's surface), $V$ the cell voltage in volts, $y$ the steam fraction and $i$ the current density in A/cm². The current density depends on $X$ and $V$ only: it is a property of the electrode at each position, the same at every height of the channel. The voltage is an **input**, like the coordinates: one training covers every voltage from 1.15 to 1.35 V, and a new voltage is answered by evaluating the network, not by training again.

In [ ]:
# The scales of the problem
print(f"X = x / L, L = {pb.L_CHANNEL*1e3:.0f} mm;  Z = z / h, h = {pb.H_CHANNEL*1e3:.0f} mm;  V = {mp.V_LOW} to {mp.V_HIGH} V")
for V in (mp.V_LOW, mp.V_HIGH):
    i0 = float(mp.current_at(V, pb.Y_IN)[0])
    print(f"V = {V:.2f} V: current density at the inlet {i0:.2f} A/cm²; profile across the channel about beta i = {mp.BETA*i0:.4f}; "
          f"drop through the electrode K_E i = {mp.K_E*i0:.3f}")

**What you should see.** At the inlet the current density is 0.99 A/cm² at 1.15 V and 1.80 A/cm² at 1.35 V. The profile across the channel is of order $\beta i$, 0.006 to 0.010, while the steam falls by 0.4 to 0.7 along the channel and by 0.23 to 0.41 through the electrode: **the profile across the channel is two orders of magnitude smaller than the other two changes**, and it is what the requirements ask you to resolve.

## B2 · The physics the network is trained with

In the scaled variables the equation and its conditions read

$$6Z(1-Z)\,\frac{\partial y}{\partial X} = \alpha\,\frac{\partial^2 y}{\partial Z^2}, \qquad
y = 0.90 \ \text{at}\ X = 0, \qquad
\frac{\partial y}{\partial Z} = \beta\, i \ \text{at}\ Z = 0, \qquad
\frac{\partial y}{\partial Z} = 0 \ \text{at}\ Z = 1,$$

$$V = E\big(y(X, 0) - K_E\, i\big) + \frac{RT}{F}\,\mathrm{asinh}\frac{i}{2 i_0} + i\,\mathrm{ASR} \quad \text{at every}\ X$$

Each line becomes a loss term:

| loss term | where it is evaluated | what it enforces | divided by |
|---|---|---|---|
| $\mathcal{L}_\mathrm{PDE}$ | points inside $(X, Z, V)$ | the steam carried along and diffused across | $\beta$ |
| $\mathcal{L}_\mathrm{inlet}$ | points on $X = 0$ | the feed, 90 % steam | $\beta$ |
| $\mathcal{L}_\mathrm{electrode}$ | points on $Z = 0$ | the steam taken by the current | $\beta$ |
| $\mathcal{L}_\mathrm{interconnect}$ | points on $Z = 1$ | the impermeable wall | $\beta$ |
| $\mathcal{L}_\mathrm{charge}$ | the same points on $Z = 0$ | the charge transfer at the reaction site | 2 mV |
| $\mathcal{L}_\mathrm{balance}$ | quadrature columns at random $(X, V)$ | the balance of the mean, $d\bar y/dX = -K_I\, i$ | $K_I$ |

**Why each is divided.** The profile across the channel is of order $\beta$ per A/cm², and so are the residuals of the equation and of the walls: divided by $\beta$ they are of order one. The charge transfer is in volts; an error of 2 mV moves the current density by about 0.5 %, and the steam at the reaction site by about 1e-03 - the size of the requirement - so 2 mV is its natural unit here (Ex10.2 used 50 mV for a looser accuracy). The balance is divided by $K_I$, the fall of the mean per A/cm².

**The balance of the mean.** Integrated over the height with the flow's weight $6Z(1-Z)$ and both wall conditions, the equation gives $d\bar y/dX = -K_I\,i$. A network that satisfied the equation and the walls exactly would satisfy it too; a trained one does not, and Ex09.2 and Ex08 showed that holding a balance by its own loss term makes a network tens of times more exact. Note what it does **not** hold: the value of the mean. Only its slope is in the term, so the mean at the inlet must come from the inlet condition, or be built in.

**What can be built in rather than trained.** Three things are known before any training:

* the **inlet value of the mean**, 0.90, which a factor $X$ can build in exactly;
* the **shape of the profile across the channel**: for a wall flux that changes slowly along the channel it is Ex09.2's fully developed profile, $y = \bar y - \beta\, i\, \varphi(Z)$ with $\varphi = Z^3 - Z^4/2 - Z + 13/35$ (`mp.phi`), whose slope is $-1$ at the electrode, $0$ at the interconnect, and whose flow-weighted mean is zero;
* the **range of the current density**: positive, and below the limiting current at the surface.

The plug-flow hand calculation of section A4 cannot be built in directly: it is an ordinary differential equation along $X$, solved by a march (`mp.plug_flow(V)`), not a formula. The cell below draws it with Ex10.2's one-dimensional model.

In [ ]:
# The hand calculation (plug flow, electrode resolved) against Ex10.2's 1-D model with its gas-transport loss
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for V, c in zip(mp.V_TEST, ("tab:blue", "tab:green", "tab:red")):
    X, ybar, i, ysite = mp.plug_flow(V)
    X1, y1, i1 = mp.one_d(V, gas_loss=True)
    ax[0].plot(100 * X, ybar, color=c, label=f"{V:.2f} V, mean (plug flow)")
    ax[0].plot(100 * X, ysite, color=c, ls=":", label=f"{V:.2f} V, reaction site")
    ax[1].plot(100 * X, i, color=c, label=f"{V:.2f} V, electrode resolved")
    ax[1].plot(100 * X1, i1, color=c, ls="--", lw=0.9, label=f"{V:.2f} V, Ex10.2's 1-D model")
    print(f"{V:.2f} V: cell current {mp.faraday(ybar[-1]):.2f} A by the hand calculation, {mp.faraday(y1[-1]):.2f} A by Ex10.2's 1-D model; "
          f"steam at the reaction site at the outlet {ysite[-1]:.3f}")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("steam fraction [-]"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("x [mm]"); ax[1].set_ylabel("current density [A/cm²]"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

**What you should see.** Left, the mean steam fraction falling along the channel, and the reaction site (dotted) far below it: at 1.33 V the reaction site sees 0.04 steam at the outlet, close to starving. Right, the current density falling along the channel as the steam does. The hand calculation gives cell currents of 93.97, 122.41 and 139.22 A; Ex10.2's 1-D model 102.48, 130.92 and 146.57 A, 5 to 9 % more, because its gas-transport loss leaves out the hydrogen that must leave the electrode. The hand calculation is close to the truth (part C) but not close enough for the requirements.

## B3 · The training points

A physics-informed network needs no solution to train on, only **points** at which the residuals are evaluated. `mp102a.py` provides two samplers, Ex09.2's on this range of voltages:

* `mp.sample_channel(n, seed)`: $n$ points $(X, Z, V)$ inside, a Latin hypercube, with a fifth of them in the first 5 % of the length, where the profile across the channel develops from the flat inlet;
* `mp.sample_faces(n, seed)`: $n$ points on each of the inlet, the electrode's surface and the interconnect, with the same concentration near the inlet on the two walls.

The voltage is drawn uniformly from 1.15 to 1.35 V. The three **test voltages, 1.18, 1.27 and 1.33 V**, are inside the range, but no point is placed at them on purpose: the network is scored where it was not fitted.

In [ ]:
# The training points
pts = mp.sample_channel(4000)                            # (X, Z, V) inside
inlet, electrode_pts, interconnect = mp.sample_faces(1000)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].scatter(100 * pts[:, 0], pts[:, 1], s=1, alpha=0.4, label="inside")
ax[0].scatter(100 * electrode_pts[:, 0], electrode_pts[:, 1], s=3, label="electrode's surface")
ax[0].scatter(100 * interconnect[:, 0], interconnect[:, 1], s=3, label="interconnect")
ax[0].scatter(100 * inlet[:, 0], inlet[:, 1], s=3, label="inlet")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("Z = z/h"); ax[0].legend(fontsize=7, loc="center right")
ax[1].hist(pts[:, 2], bins=40)
for V in mp.V_TEST:
    ax[1].axvline(V, color="red", lw=1)
ax[1].set_xlabel("cell voltage V [V]"); ax[1].set_ylabel("points")
plt.tight_layout(); plt.show()

**What you should see.** Left, the channel (drawn in scaled height) with its points, denser in the first 5 mm; right, a flat distribution of voltages with the three test voltages in red.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the solver an engineer would run: the march on an engineering grid | the same scheme on a grid fine enough to be a hundred times more exact than anything scored |
| grid | 21 × 5 nodes (along × across) | 1601 × 161 nodes |
| its job | to be **timed** against the network: the online cost of answering a new voltage | to **score** the network and the reference |

Both march the same discrete equations, written in `mp102a.March`. This part explains the scheme, builds confidence in the ground truth, and measures the reference model's accuracy and time - the bar part D has to clear.

## C2 · The scheme, step by step

1. **Nodes.** A grid of $n_x \times n_z$ nodes covers the channel, walls included.
2. **Across the channel.** At every node the second derivative across is the central difference $(y_{j+1} - 2y_j + y_{j-1})/h_Z^2$.
3. **The walls by ghost nodes.** The interconnect's ghost mirrors its neighbour (zero slope); the electrode's carries the flux $\beta\, i$, which depends on the current density at that $X$.
4. **The current by Newton's method.** At every step the current density is the root of the charge transfer at the reaction site, $V = E(y_0 - K_E\,i) + \dots$, given the steam $y_0$ at the wall node: Newton's method kept inside the interval $0 < i < y_0/K_E$, where the root is unique. Its derivative $di/dy_0$ comes with it.
5. **Along the channel: Crank–Nicolson.** $X$ is stepped like time: each step averages the right-hand side over the old and the new column, which is second order. It needs one sparse solve per step with a matrix factorised once.
6. **The nonlinear wall: Newton's method on one node.** Only the wall row depends nonlinearly on the unknown, through $i(y_0)$, so each step linearises it about the last estimate and solves again; the rank-one change of the matrix is handled without refactorising (Sherman–Morrison).
7. **The start: Rannacher's.** At the walls the velocity is zero, so a wall row has no marching term, and Crank–Nicolson would hold the wall condition only as the average of two columns. The flat inlet breaks it, and Ex09.2 found the error then flips sign at every step and never decays. The first two columns are therefore taken as two fully implicit half steps each.
8. **The flow carried exactly.** The trapezoid rule on $n_z$ nodes integrates $6Z(1-Z)$ to $1 - h_Z^2$, not to 1, so the velocity row is divided by that integral: the grid carries exactly the flow fed, and the balance of the mean holds on any grid (MP9.2A).
9. **The cell current from the steam converted.** By Faraday's law, $I = (0.90 - \bar y_\mathrm{out})/K_I \times 100\ \mathrm{cm^2}$. The current density integrated along the channel gives the same, but converges more slowly: near the inlet it changes like $\sqrt{x}$ as the profile develops, which the trapezoid rule integrates at order 1.5. The difference of the two is the conservation check.

`mp.March(nx, nz).run(V)` returns the field $y$ (nz, nx) and the current density $i$ (nx); `mp.reference(V)` runs it on the engineering grid and returns the current, the outlet mean and the reaction-site steam along the channel.

In [ ]:
# The ground truth: its convergence study at 1.33 V, from the truth file
G = np.load("MP10.2A_truth.npz")
print("grid (nx × nz)   cell current [A]   outlet mean [-]   reaction site at the outlet [-]   trapezoid against Faraday [-]")
for g, I, m, s, b in zip(G["conv_grids"], G["conv_current"], G["conv_mean_out"], G["conv_site_out"], G["conv_balance"]):
    print(f"{int(g[0]):5d} × {int(g[1]):3d}       {I:.6f}         {m:.7f}         {s:.8f}                     {abs(b):.1e}")
for q, name in (("current", "cell current [A]"), ("mean_out", "outlet mean [-]"), ("site_out", "reaction site at the outlet [-]")):
    d = np.abs(np.diff(G[f"conv_{q}"]))
    print(f"{name}: change between grids", " ".join(f"{v:.1e}" for v in d), "; observed order",
          " ".join(f"{np.log2(d[j] / d[j + 1]):.2f}" for j in range(len(d) - 1)))
print(f"Richardson: cell current {float(G['conv_current_extrapolated']):.4f} A, outlet mean {float(G['conv_mean_out_extrapolated']):.6f}")

**What you should see.** Five grids, each twice as fine along and across. The changes between grids fall by four each time: the cell current by 3.3e-03, 8.3e-04, 2.1e-04 and 5.2e-05 A, the outlet mean by 1.5e-05 to 2.4e-07, with an **observed order of 2.00**, the scheme's; the reaction site at the outlet changes by less than 3e-07, approaching order 2 (1.78, 1.95, 1.99). The ground truth's grid, 1601 × 161, is within 2e-05 A (1.2e-07 relative) of the Richardson extrapolation in the current and within 1e-07 in the outlet mean: about ten thousand times finer than the requirements. The trapezoid integral of the current density agrees with Faraday's law to 3.1e-07, the difference falling at order 1.4 to 1.7 as the $\sqrt{x}$ at the inlet predicts: the steam is conserved.

In [ ]:
# The zero-strength check, run here: both transport resistances times S, against the 1-D model of infinitely fast transport
rows = mp.check_zero_strength()
print("ratios of the current difference, S = 0.1 to 0.01 to 0.001:", " ".join(f"{rows[k][1] / rows[k + 1][1]:.2f}" for k in range(2)))

**What you should see.** At 1.29 V the one-dimensional model of infinitely fast transport gives 153.42 A. With both resistances a tenth of this project's ($S$ = 0.1) the march gives 150.38 A, at $S$ = 0.01 153.10 A, at $S$ = 0.001 153.39 A: the difference falls from 3.04 to 0.32 to 0.03 A, **in proportion to $S$** (ratios 9.6 and 9.9), and so does the outlet mean's. The march reduces to the one-dimensional model where it must: the code is right in the limit where the answer is known independently.

## C3 · The reference model: accuracy and time

The reference grid was chosen as the coarsest that meets all three requirements at every voltage of the range (21 voltages from 1.15 to 1.35 V): 11 × 3 misses the current (0.22 %), 21 × 3 misses the current and the mean (0.10 %, 6.8e-04), **21 × 5 meets all three** (at most 0.05 %, 3.2e-04 and 1.1e-04). The cell below scores it against the ground truth at the test voltages and times it. It runs on the CPU of the runtime: a march of 21 small solves gains nothing from a GPU, and an engineer would not run it on one.

In [ ]:
# The reference model against the ground truth, at the three test voltages
worst = np.zeros(3)
for V in mp.V_TEST:
    r = mp.reference(V); k = f"{V:.2f}"
    truth_site = np.interp(r["X"], G["X_line"], G[f"y_site_{k}"])            # the ground truth at the reference's nodes
    e = np.array([abs(r["current"] / float(G[f"current_{k}"]) - 1), abs(r["mean_out"] - float(G[f"mean_out_{k}"])),
                  np.abs(r["y_site"] - truth_site)[r["X"] >= mp.X_SCORE].max()])
    worst = np.maximum(worst, e)
    print(f"{V:.2f} V: cell current {r['current']:.2f} A (truth {float(G[f'current_{k}']):.2f} A, {100*e[0]:.3f} %); "
          f"outlet mean {r['mean_out']:.4f} (truth {float(G[f'mean_out_{k}']):.4f}); reaction site worst error {e[2]:.2e}")
t_ref = middle(lambda: mp.reference(1.24))               # the online cost of a new voltage
print(f"\nreference model: worst {100*worst[0]:.3f} % in the current, {worst[1]:.2e} in the outlet mean, {worst[2]:.2e} at the reaction site; "
      f"{t_ref*1e3:.2f} ms per new voltage")
print(f"the network must answer a new voltage in under {t_ref*1e3/10:.2f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "worst_current": worst[0], "worst_mean": worst[1], "worst_site": worst[2]}   # kept for part E

**What you should see.** At every test voltage the reference model is inside the requirements: the current within 0.043 to 0.049 %, the outlet mean within 3.1e-04, the reaction site within 1.1e-04. One march takes a few milliseconds (4 to 11 ms on the authors' CPU, shared with other jobs). **The reference model is cheap**: 21 small linear solves with a few Newton steps each. That sets a hard bar for the network's online time, which part D measures on this same runtime.

In [ ]:
# The ground truth at the three test voltages: the steam along the channel, and across it at the outlet
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for V, c in zip(mp.V_TEST, ("tab:blue", "tab:green", "tab:red")):
    k = f"{V:.2f}"; X = G["X_line"]
    ax[0].plot(100 * X, G[f"mean_{k}"], color=c, label=f"{V:.2f} V, mean")
    ax[0].plot(100 * X, G[f"y_wall_{k}"], color=c, ls="--", lw=0.8)
    ax[0].plot(100 * X, G[f"y_site_{k}"], color=c, ls=":", label=f"{V:.2f} V, reaction site")
    ax[1].plot(100 * X, G[f"i_{k}"], color=c, label=f"{V:.2f} V")
    f = G[f"field_{k}"][:, -1]
    ax[2].plot(1e3 * (f - float(G[f"mean_out_{k}"])), G["Z"], color=c, label=f"{V:.2f} V")
ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("steam fraction [-]"); ax[0].legend(fontsize=7); ax[0].set_title("dashed: the electrode's surface", fontsize=9)
ax[1].set_xlabel("x [mm]"); ax[1].set_ylabel("current density [A/cm²]"); ax[1].legend(fontsize=7)
ax[2].set_xlabel("steam fraction at the outlet minus its mean [1e-3]"); ax[2].set_ylabel("Z = z/h"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
plug = G["plug_errors_sweep"]
print("the plug-flow hand calculation over 21 voltages: worst", f"{100*np.abs(plug[:, 0]).max():.2f} % in the current,",
      f"{np.abs(plug[:, 1]).max():.2e} in the outlet mean, {plug[:, 2].max():.2e} at the reaction site")
over = 100 * (G["current_sweep_ex102_1d"] / G["current_sweep"] - 1)
print(f"Ex10.2's 1-D model overstates the cell current by {over.min():.1f} to {over.max():.1f} % over the range")

**What you should see.** Left, the mean steam fraction falling along the channel, the electrode's surface (dashed) just below it and the reaction site (dotted) far below: at the outlet 0.29, 0.12 and 0.04 at the three voltages. Middle, the current density falling along the channel, from 1.11, 1.46 and 1.71 A/cm² at the inlet to 0.79, 0.97 and 0.98 at the outlet. Right, the profile across the channel at the outlet: lowest at the electrode's surface, 1.7 to 2.1 thousandths below the mean, highest at the interconnect, 0.6 to 0.7 thousandths above it. The plug-flow hand calculation misses by up to 0.19 % in the current, 1.27e-03 in the outlet mean and 3.01e-03 at the reaction site: the network has to resolve the profile across the channel. Ex10.2's one-dimensional model overstates the current by 4.9 to 9.9 %.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `y, i = net(X, Z, V)`:

| argument | meaning | shape |
|---|---|---|
| `X`, `Z` | $x/L$ and $z/h$, in 0 to 1 ($Z = 0$ the electrode's surface) | `(n, 1)` |
| `V` | the cell voltage in volts, 1.15 to 1.35 | `(n, 1)` |
| returns `y` | the steam fraction at $(X, Z)$ | `(n, 1)` |
| returns `i` | the current density at $X$ in A/cm², the same for every $Z$ | `(n, 1)` |

Whatever you compute inside - the fully developed profile, features, several sub-networks - stays inside `net`. The loss below and the evaluation at the end only call `net(X, Z, V)`.

## D2 · The loss, given

`losses(net, seed)` draws training points with the samplers of part B (the same points for the same seed, as L-BFGS needs) and returns the five residual terms of section B2's table: the equation inside, the inlet, the electrode's surface and the interconnect, each divided by $\beta$, and the charge transfer at the reaction site in units of 2 mV, at the electrode's points. `balance_loss(net, seed)` is the balance of the mean at 300 random $(X, V)$, the mean taken by 16-point Gauss–Legendre quadrature across the height with the flow's weight, its slope along $X$ by autograd, divided by $K_I$. Two details:

* **The current density is read at the electrode's points** (and at $Z$ = 0 in the balance), where the charge transfer holds; your network must return the same `i` at any `Z`.
* **The inlet residual cannot vanish at the inlet corner**, where the flat feed meets the electrode's flux: the exact field is singular there. Give it a small weight, or it dominates the loss (MP8.2A's lesson).

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a, grad=False):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1).requires_grad_(grad)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

CHARGE_SCALE = 0.002                                     # V: the charge transfer's residual in units of 2 mV

def losses(net, seed=88, n_in=4000, n_face=1000):
    # inside: 6 Z (1 - Z) y_X = alpha y_ZZ, divided by beta
    p = mp.sample_channel(n_in, seed)
    X, Z, V = T(p[:, 0], True), T(p[:, 1], True), T(p[:, 2])
    y, _ = net(X, Z, V)
    r_pde = (6 * Z * (1 - Z) * d(y, X) - mp.ALPHA * d(d(y, Z), Z)) / mp.BETA
    inlet, el, ic = mp.sample_faces(n_face, seed)
    # the inlet, X = 0: y = 0.90, divided by beta
    yi, _ = net(T(inlet[:, 0]), T(inlet[:, 1]), T(inlet[:, 2]))
    r_in = (yi - pb.Y_IN) / mp.BETA
    # the electrode's surface, Z = 0: y_Z = beta i, divided by beta
    Ze, Ve = T(el[:, 1], True), T(el[:, 2])
    ye, ie = net(T(el[:, 0]), Ze, Ve)
    r_el = d(ye, Ze) / mp.BETA - ie
    # the charge transfer at the reaction site, y_r = y(X, 0) - K_E i, in units of 2 mV
    r_ch = (mp.v_site(ie, ye - mp.K_E * ie) - Ve) / CHARGE_SCALE
    # the interconnect, Z = 1: y_Z = 0, divided by beta
    Zc = T(ic[:, 1], True)
    yc, _ = net(T(ic[:, 0]), Zc, T(ic[:, 2]))
    r_ic = d(yc, Zc) / mp.BETA
    return {"pde": r_pde.pow(2).mean(), "inlet": r_in.pow(2).mean(), "electrode": r_el.pow(2).mean(),
            "interconnect": r_ic.pow(2).mean(), "charge": r_ch.pow(2).mean()}

# the balance of the mean (section B2): d ybar / dX = -K_I i, divided by K_I
_zq, _wq = np.polynomial.legendre.leggauss(16)            # Gauss-Legendre nodes and weights on [-1, 1]
ZQ = T((_zq + 1) / 2)                                    # moved to [0, 1]
WQ = T(_wq / 2 * 6 * ((_zq + 1) / 2) * (1 - (_zq + 1) / 2)).ravel()   # the weights carry the flow's 6 Z (1 - Z); they sum to 1

def balance_loss(net, seed=7, n=300):
    q = mp.sample_channel(n, seed)[:, [0, 2]]             # (X, V) pairs
    X = T(np.repeat(q[:, 0], 16), True)                  # each pair once per quadrature node
    V = T(np.repeat(q[:, 1], 16))
    y, _ = net(X, ZQ.repeat(n, 1), V)
    rate = (d(y, X).reshape(n, 16) * WQ).sum(1, keepdim=True)   # d ybar / dX at each (X, V), by autograd
    _, i = net(T(q[:, 0]), torch.zeros(n, 1, device=DEVICE), T(q[:, 1]))   # the current density at each (X, V)
    return ((rate + mp.K_I * i) / mp.K_I).pow(2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The inputs scaled to order one ($2X - 1$, $2Z - 1$, the voltage mapped to $[-1, 1]$), or features built from them;
* **what is built in.** The inlet value of the mean, the fully developed profile `mp.phi(Z)` times $\beta\,i$, the range of the current density (section B2);
* **its size.** One network with two outputs, or one for $(X, V)$ and one for the profile; layers, neurons, activation. Remember why Part 2 uses tanh, not ReLU.

What the problem tells you, from parts A to C: the mean falls by up to 0.66 along the channel, the current density changes by a factor of three along it and with the voltage, and the profile across the channel is a few thousandths. A single network asked to return all of it in $y$ must resolve one part in a thousand of its own output, and the balance term holds only the slope of the mean, not its value.

In [ ]:
# TODO 1 - your network. y, i = net(X, Z, V), every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, X, Z, V):
        ...                                              # return y and i, each of shape (n, 1)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` and `balance_loss`. Decide the optimiser or optimisers, the number of steps, the learning rate, the weight of each term, how many points and whether they are drawn afresh. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

The course's recipe (Adam on fresh points, then L-BFGS on one fixed set of points) is a good start. `quick_score()` below gives the three errors at the test voltages against the ground truth - the worst relative error of the cell current in per cent, the worst error of the outlet mean, and the worst error of the reaction-site steam beyond 5 mm - in a fraction of a second.

In [ ]:
# A quick score, given: the three errors at the test voltages against the ground truth
G = np.load("MP10.2A_truth.npz")
def answer(V, X=np.linspace(0, 1, 401)):
    """The network's answer at one voltage: the cell current [A] (trapezoid of i), the outlet mean, and the
    reaction-site steam on the points X."""
    with torch.no_grad():
        yw, i = net(T(X), T(np.zeros_like(X)), T(np.full_like(X, V)))
        yq, _ = net(torch.ones(16, 1, device=DEVICE), ZQ, torch.full((16, 1), V, device=DEVICE))
    yw, i = yw.cpu().numpy().ravel(), i.cpu().numpy().ravel()
    return np.trapezoid(i, X) * pb.A_CELL, float((yq.ravel() * WQ).sum()), yw - mp.K_E * i

def quick_score():
    worst = np.zeros(3); X = G["X_line"]
    for V in mp.V_TEST:
        k = f"{V:.2f}"; I, m, site = answer(V, X)
        worst = np.maximum(worst, [100 * abs(I / float(G[f"current_{k}"]) - 1), abs(m - float(G[f"mean_out_{k}"])),
                                   np.abs(site - G[f"y_site_{k}"])[X >= mp.X_SCORE].max()])
    return f"current {worst[0]:.3f} %, outlet mean {worst[1]:.2e}, reaction site {worst[2]:.2e}"

In [ ]:
# TODO 2 - your training
history = {"pde": [], "inlet": [], "electrode": [], "interconnect": [], "charge": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; {quick_score()}")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the steam at the reaction site along the channel at one test voltage against the ground truth, with its error. Save each figure you want in the report as `MP10.2A_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* at the test voltages 1.18, 1.27 and 1.33 V: the cell current (the trapezoid of $i$ on 1601 points), the outlet mean (the quadrature of `balance_loss`), and the steam at the electrode's surface, at the reaction site and the current density along the channel (1601 points);
* the same at 1.12 and 1.38 V, outside the trained range, and the cell current and outlet mean at the 21 voltages of the ground truth's sweep (the polarisation curve);
* the field at 1.27 V on the ground truth's 401 × 41 nodes;
* the **online time** of a new voltage on this runtime - the current, the outlet mean and the reaction-site steam on 101 points, from one call of `net` - and the same for 10 and 100 voltages at once, against the reference model run 1, 10 and 100 times.

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
Xl = G["X_line"]
def online(voltages, X=np.linspace(0, 1, 101)):
    # the cell current, the outlet mean and the reaction-site steam at several voltages, from one call of the network
    voltages = np.atleast_1d(voltages); n, k = len(X), len(voltages)
    Xa = np.r_[np.tile(X, k), np.ones(16 * k)]                                 # the electrode's surface, then the outlet's quadrature column
    Za = np.r_[np.zeros(n * k), np.tile(ZQ.cpu().numpy().ravel(), k)]
    Va = np.r_[np.repeat(voltages, n), np.repeat(voltages, 16)]
    with torch.no_grad():
        y, i = net(T(Xa), T(Za), T(Va))
    yw, iw = y[: n * k].reshape(k, n), i[: n * k].reshape(k, n)
    current = torch.trapezoid(iw, torch.as_tensor(X, device=DEVICE), dim=1) * pb.A_CELL
    mean_out = (y[n * k:].reshape(k, 16) * WQ).sum(1)
    return current.cpu().numpy(), mean_out.cpu().numpy(), (yw - mp.K_E * iw).cpu().numpy()

out = {}
for V in tuple(mp.V_TEST) + tuple(G["V_outside"]):
    k = f"{V:.2f}"
    with torch.no_grad():
        yw, i = net(T(Xl), T(np.zeros_like(Xl)), T(np.full_like(Xl, V)))
    yw, i = yw.cpu().numpy().ravel(), i.cpu().numpy().ravel()
    out[f"y_wall_{k}"], out[f"i_{k}"], out[f"y_site_{k}"] = yw, i, yw - mp.K_E * i
    out[f"current_{k}"] = np.array(np.trapezoid(i, Xl) * pb.A_CELL)
    out[f"mean_out_{k}"] = np.array(online(V)[1][0])
out["current_sweep"], out["mean_out_sweep"], _ = online(G["V_sweep"])
XX, ZZ = np.meshgrid(G["X"], G["Z"])
with torch.no_grad():
    out["field_1.27"] = net(T(XX), T(ZZ), T(np.full(XX.size, 1.27)))[0].reshape(XX.shape).cpu().numpy()

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    vs = np.linspace(1.151, 1.349, k)
    timing[f"net_{k}"] = middle(lambda: online(vs))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(v) for v in vs], reps=3 if k == 100 else 5)
out.update({key: np.array(v) for key, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP10.2A_net.pt"))   # the trained weights, beside the report
for k in (1, 10, 100):
    print(f"online, {k:3d} new voltage(s): network {timing[f'net_{k}']*1e3:7.2f} ms, reference {timing[f'ref_{k}']*1e3:7.2f} ms "
          f"({timing[f'ref_{k}']/timing[f'net_{k}']:.1f} times)")
print(quick_score())

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's times `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the twelve questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP10.2A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): worst {100*REF['worst_current']:.3f} % in the current, {REF['worst_mean']:.2e} in the outlet mean, "
      f"{REF['worst_site']:.2e} at the reaction site; {REF['t_ref_s']*1e3:.2f} ms per new voltage")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* at each test voltage, the relative error of the cell current in per cent, the error of the outlet mean, and the worst error of the reaction-site steam beyond 5 mm;
* the largest error of the field at 1.27 V beyond 5 mm;
* the network's online time for one new voltage, the reference model's on the same runtime, and their ratio;
* the offline training time.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP10.2A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the error of the reaction-site steam along the channel at the three test voltages; the error of the field at 1.27 V; the polarisation curve - the cell current against the voltage (`R["current_sweep"]` against `G["current_sweep"]`, with Ex10.2's 1-D model, `G["current_sweep_ex102_1d"]`); and the online times of the network and the reference model for 1, 10 and 100 voltages (`R["net_1"]`, `R["ref_10"]` and so on).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the twelve questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 7 are about your network and its result, 8 to 11 ask you to use it, 12 concludes. The next cell writes `MP10.2A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features or structure built in), sub-networks, layers, neurons and number of weights. Why this design?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training, and the weight you gave it.": """
""",
    "3. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time on the T4 in seconds.": """
""",
    "4. The cell current at 1.18, 1.27 and 1.33 V: your errors in per cent against the ground truth. Are they within 0.1 %? Compare them with the plug-flow hand calculation's and the reference model's errors.": """
""",
    "5. The outlet's mixing-cup steam fraction at the three test voltages: your errors. Are they within 5e-04? How did you make sure the network's mean starts at exactly 0.90?": """
""",
    "6. The steam at the reaction site beyond the first 5 mm: your worst error at each test voltage, and where along the channel it is largest. What happens in the first 5 mm, in the ground truth and in your network?": """
""",
    "7. The online time to answer a new voltage: your network against the reference model on the same runtime, in milliseconds, and the ratio; the same for 10 and 100 voltages at once. What does each method spend its time on?": """
""",
    "8. At 1.33 V and the outlet: by how much does the electrode's surface lie below the mixing-cup mean, and the reaction site below the surface? Which of the two transport steps costs the cell more current, and how do you know?": """
""",
    "9. The polarisation curve: how far is your network's cell current from the ground truth over the 21 voltages, and by how many per cent does Ex10.2's 1-D model overstate it? Explain the difference from the physics of the electrode.": """
""",
    "10. Outside the range: your errors at 1.12 and 1.38 V against the ground truth (current, outlet mean, reaction site). How fast does the error grow outside the range, and why?": """
""",
    "11. Train again with another seed. How much do the three errors and the training time of the two runs differ? Is your result reproducible within the requirements?": """
""",
    "12. To conclude: for which engineering task would you use your network rather than the reference march, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP10.2A — The Channel Across Its Height, and the Electrode Under It", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP10.2A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP10.2A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP10.2A_report.md into MP10.2A_report.pdf, with any figure
# saved as MP10.2A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP10.2A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP10.2A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP10.2A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP10.2A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass